### RAG pipeline


### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [46]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [47]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../Input Files")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 82 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 174 0 (offset 0)
Ignoring wrong pointing object 180 0 (offset 0)
Ignoring wrong pointing object 187 0 (offset 0)
Ignoring wrong pointing object 196 0 (offset 0)
Ignoring wrong pointing object 231 0 (offset 0)
Ignoring wrong pointing object 244 0 (offset 0)
Ignoring wrong pointing object 252 0 (offset 0)
Ignoring wrong pointing object 254 0 (offset 0)
Ignorin

Found 3 PDF files to process

Processing: 16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf
  ✓ Loaded 37 pages

Processing: Global and local indicators of spatial connectivity for.pdf
  ✓ Loaded 34 pages

Processing: Saturn paper.pdf
  ✓ Loaded 43 pages

Total documents loaded: 114


In [48]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2023-04-19T16:59:29+00:00', 'author': 'Catherine Gao-Howard', 'moddate': '2023-05-24T20:29:44-04:00', 'title': 'CarpeDiem_Supplement_revised_acceptedChanges', 'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf'}, page_content='1 \nSupplemental Materials \n \nMachine learning links unresolving secondary pneumonia to mortality in patients with severe \npneumonia, including COVID-19 \n \nCatherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth \nNannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 , James M. Walter1 , Jacqueline M. Kruser1,4, Luke \nRasmussen5, Daniel Schneider3, Justin Starren5, Helen K. Donnelly1, Alvaro Donayre1, Yuan Luo5, GR Scott \nBudinger1,6 **, Ric

In [49]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [50]:
chunks=split_documents(all_pdf_documents)
chunks

Split 114 documents into 292 chunks

Example chunk:
Content: 1 
Supplemental Materials 
 
Machine learning links unresolving secondary pneumonia to mortality in patients with severe 
pneumonia, including COVID-19 
 
Catherine A. Gao1 *, Nikolay S. Markov1 *, Th...
Metadata: {'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2023-04-19T16:59:29+00:00', 'author': 'Catherine Gao-Howard', 'moddate': '2023-05-24T20:29:44-04:00', 'title': 'CarpeDiem_Supplement_revised_acceptedChanges', 'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2023-04-19T16:59:29+00:00', 'author': 'Catherine Gao-Howard', 'moddate': '2023-05-24T20:29:44-04:00', 'title': 'CarpeDiem_Supplement_revised_acceptedChanges', 'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf'}, page_content='1 \nSupplemental Materials \n \nMachine learning links unresolving secondary pneumonia to mortality in patients with severe \npneumonia, including COVID-19 \n \nCatherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth \nNannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 , James M. Walter1 , Jacqueline M. Kruser1,4, Luke \nRasmussen5, Daniel Schneider3, Justin Starren5, Helen K. Donnelly1, Alvaro Donayre1, Yuan Luo5, GR Scott \nBudinger1,6 **, Ric

### embedding And vectorStoreDB

In [51]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [52]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_29308\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [53]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 292


In [54]:
chunks

[Document(metadata={'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'creator': 'Word', 'creationdate': '2023-04-19T16:59:29+00:00', 'author': 'Catherine Gao-Howard', 'moddate': '2023-05-24T20:29:44-04:00', 'title': 'CarpeDiem_Supplement_revised_acceptedChanges', 'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'total_pages': 37, 'page': 0, 'page_label': '1', 'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf'}, page_content='1 \nSupplemental Materials \n \nMachine learning links unresolving secondary pneumonia to mortality in patients with severe \npneumonia, including COVID-19 \n \nCatherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth \nNannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 , James M. Walter1 , Jacqueline M. Kruser1,4, Luke \nRasmussen5, Daniel Schneider3, Justin Starren5, Helen K. Donnelly1, Alvaro Donayre1, Yuan Luo5, GR Scott \nBudinger1,6 **, Ric

In [55]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 292 texts...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Generated embeddings with shape: (292, 384)
Adding 292 documents to vector store...
Successfully added 292 documents to vector store
Total documents in collection: 584


### Retriever Pipeline From VectorStore

In [56]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -1.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Chroma's cosine distance is converted to cosine similarity with
        ``similarity = 1 - distance``. Cosine similarity can range from -1 to 1,
        so the default threshold must not exclude negative but valid results.
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            documents = (results.get('documents') or [[]])[0]
            metadatas = (results.get('metadatas') or [[]])[0]
            distances = (results.get('distances') or [[]])[0]
            ids = (results.get('ids') or [[]])[0]

            for rank, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances), start=1
            ):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'distance': distance,
                        'rank': rank
                    })

            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents met the score threshold")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [57]:
rag_retriever

In [58]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_7c242483_245',
  'content': 'operates by observing targets through two different field of views, the preceding (FoVP) and fol-\nlowing (FoVF) fields of view, which are analyzed separately. To model the𝜓dependence, we fitted\nquadratic functions to the GSA fluxes (Fig. S1B). The overall trend of GSA magnitude as a function\nof𝜁is described by linear models. There is a brightness dip at𝜁∼−0.2 for FoVP (Fig. S1A), and the\n𝜁values around the dip correspond to a region near the gap between CCD rows 5 and 6 (Fig. S1E).\nThe FoVF data near the corresponding CCD gap, at𝜁∼−0.1 (Fig. S1F), show a brightness decrease\nconsistent with a dip, but the sparser coverage compared with FoVP prevents fully characterizing\nthe feature. We ascribe the observed dip to the effect of the detector gap. We model the FoVP dip\nas an extra Gaussian component, whose amplitude is expressed in magnitude, added to the linear\nmodel of magnitude. We jointly fitted of the𝜁and𝜓models to the GSA baseline (F

In [59]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c30e6c6a_31',
  'content': 'Reduction [Internet]. arXiv [stat.ML] 2018;http://arxiv.org/abs/1802.03426. cited \n15. Chen T, Guestrin C. XGBoost: A Scalable Tree Boosting System. In: Proceedings of the 22nd ACM \nSIGKDD International Conference on Knowledge Discovery and Data Mining. New York, NY, USA: Association \nfor Computing Machinery; 2016:785–794 \n16. Pickens CI et al. An adjudication protocol for severe bacterial and viral pneumonia [Internet]. bioRxiv 2022; \ndoi:10.1101/2022.10.26.22281461',
  'metadata': {'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext',
   'moddate': '2023-05-24T20:29:44-04:00',
   'title': 'CarpeDiem_Supplement_revised_acceptedChanges',
   'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf',
   'total_pages': 37,
   'content_length': 471,
   'author': 'Catherine Gao-Howard',
   'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf',
   'file_type': 'pdf',
   'page_label': '11',
   'doc_index': 31

### RAG Pipeline- VectorDB To LLM Output Generation

In [60]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [61]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [62]:
class GroqLLM:
    def __init__(self, model_name: str ="openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [64]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c30e6c6a_31',
  'content': 'Reduction [Internet]. arXiv [stat.ML] 2018;http://arxiv.org/abs/1802.03426. cited \n15. Chen T, Guestrin C. XGBoost: A Scalable Tree Boosting System. In: Proceedings of the 22nd ACM \nSIGKDD International Conference on Knowledge Discovery and Data Mining. New York, NY, USA: Association \nfor Computing Machinery; 2016:785–794 \n16. Pickens CI et al. An adjudication protocol for severe bacterial and viral pneumonia [Internet]. bioRxiv 2022; \ndoi:10.1101/2022.10.26.22281461',
  'metadata': {'creator': 'Word',
   'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf',
   'source': '..\\Input Files\\pdf_files\\16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf',
   'page': 10,
   'author': 'Catherine Gao-Howard',
   'page_label': '11',
   'file_type': 'pdf',
   'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext',
   'content_length': 471,
   'creationdate': '2023-04-19T16:59:29+00:00',
   'total_pages': 37,
   'moddate': '2023-05-24T20:29:44-0

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [66]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: -1.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
The attention mechanism is a component in neural‑network models that lets the system dynamically focus on the most relevant parts of the input when producing each part of the output. Instead of treating all input elements equally, it computes a set of attention weights (often via similarity scores) that highlight important features or positions, and then combines the input representations weighted by these scores. This enables the model to capture long‑range dependencies and improves performance in tasks such as machine translation, image captioning, and many other sequence‑to‑sequence or multimodal problems.


### Enhanced RAG Pipeline Features

In [67]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("An object in the Einstein desert", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'An object in the Einstein desert'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Answer: An “Einstein‑desert” object is a microlensing lens whose angular Einstein radius falls in the sparsely populated gap between the very small radii of free‑floating‑planet events (θ_E ≲ 9 µas) and the larger radii of brown‑dwarf events (θ_E ≳ 25 µas).  

The event KMT‑2024‑BLG‑0792 / OGLE‑2024‑BLG‑0516 has  
* θ_E ≈ 18.6 µas, placing it squarely in this desert,  
* a source star in the Galactic bulge (D_s ≈ 7.9 kpc), and  
* a lens at D_l ≈ 3.0 kpc with a relative parallax π_rel ≈ 0.20 mas.  

Its inferred mass is ≈ 1 M_J, i.e., roughly a Jupiter‑mass object that could be either a high‑mass planet or a low‑mass brown dwarf, illustrating the paucity of such intermediate‑mass lenses in the Einstein‑desert region.
Sources: [{'source': 'Saturn paper.pdf', 'page': 5, 'score': 0.16752105951309204, 'preview': 'also infer the relative lens-source trigonometric parallax,𝜋 rel =0.202 +0.054\n−0.052 mas. The d

In [68]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
No documents met the score threshold

Final Answer: No relevant context found.
Summary: The reply states that there is no pertinent information available. Consequently, no contextual details can be provided.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The reply states that there is no pertinent information available. Consequently, no contextual details can be provided.'}
